# Offline pose estimation — a recorded video or image folder

The same estimator the live loop runs, driven frame by frame over a file so you can
stop and look. Use this to check a capture before trusting it, to produce the CSV a
controller replay needs, or to see *why* a particular frame failed.

For the live camera version see `online_camera.ipynb`; for what the numbers can and
cannot mean see `bounds.py` and lecture notes §13.

In [ ]:
import sys, time, json, math
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt

POSE = Path.cwd()
if POSE.name != "pose":                      # tolerate running from the repo root
    POSE = next(p for p in [POSE / "controller/pose", POSE / "ESP32_PMW/controller/pose"]
                if p.exists())
sys.path[:0] = [str(POSE), str(POSE / "validation")]

import conic, segment, estimator, zeroing, calibration, sources, recorder, bounds
from estimator import PoseEstimator, RADIUS_MM

RESULTS = POSE.parents[1] / "results" / "pose_validation"
K, dist = estimator.load_intrinsics()

print("pose package :", POSE)
print("rim radius   :", RADIUS_MM, "mm")
print("axial fit    :", segment.AXIAL_DEFAULT, " (POSE_AXIAL=0 to disable)")
print("intrinsics   :", f"f={K[0, 0]:.0f} px, principal ({K[0, 2]:.0f}, {K[1, 2]:.0f})")

## 1. Open the source

`sources.open_source` takes a video file, a directory of images, or `"camera"` /
`"camera:1"`. It yields `(capture_time, frame)` and `None` at the end.

Look at the first frame before running anything over it. The segmenter is a fixed
threshold (`segment.THRESH`, 128) on a bright robot against a dark ground — if the
histogram below does not show two clearly separated humps, no amount of downstream
maths will save it.

In [ ]:
SOURCE = "video.mp4"          # <- a video file, or a folder of frames
MAX_FRAMES = 0                # 0 = to the end

src = sources.open_source(SOURCE)
first = src.read()
assert first is not None, f"nothing readable at {SOURCE}"
t0, frame0 = first
gray0 = cv2.cvtColor(frame0, cv2.COLOR_BGR2GRAY) if frame0.ndim == 3 else frame0

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].imshow(gray0, cmap="gray"); ax[0].set_title(f"frame 0  {gray0.shape[1]}x{gray0.shape[0]}")
ax[0].axis("off")
ax[1].hist(gray0.ravel(), bins=64, color="0.4")
ax[1].axvline(segment.THRESH, color="crimson", label=f"THRESH={segment.THRESH}")
ax[1].set_yscale("log"); ax[1].legend(); ax[1].set_title("intensity")
plt.tight_layout(); plt.show()

## 2. Check the segmentation on one frame

The rim is a thin ring, so lighting routinely breaks it into arcs and the largest
connected blob becomes the blade cross in the middle — which fits an ellipse ~37 %
too small and lands straight on the depth estimate. The convex hull is what avoids
that. `fit_rms_px` is the single best number for spotting a frame where
segmentation grabbed the wrong thing.

In [ ]:
seg = segment.segment(gray0)
assert seg is not None, "no detection in frame 0"
print(f"hull points {seg.n_points}   area {seg.area_px:.0f} px   fit rms {seg.fit_rms_px:.3f} px")
print(f"ellipse     centre ({seg.ellipse[0][0]:.1f}, {seg.ellipse[0][1]:.1f})  "
      f"axes {seg.ellipse[1][0]:.1f} x {seg.ellipse[1][1]:.1f} px  angle {seg.ellipse[2]:.1f} deg")

vis = segment.draw(cv2.cvtColor(gray0, cv2.COLOR_GRAY2BGR), seg)
plt.figure(figsize=(6, 5)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis("off"); plt.title("hull + fitted rim"); plt.show()

## 3. Run the whole file

One `PoseEstimator` for the run, because it carries the branch history that
resolves the two-fold ambiguity: every frame yields two poses and no single frame
can choose between them (lecture notes §13.6). Reset it and that prior is gone.

`PoseFilter` is here for **velocity**, not smoothing — filtering position measurably
does nothing, because the residual is correlated frame to frame.

In [ ]:
from filter import PoseFilter
from recorder import PoseRecorder

est = PoseEstimator(camera_matrix=K, dist_coeffs=dist)
filt = PoseFilter()
rows, t_ms, lost = [], [], 0

with PoseRecorder(str(POSE / "offline_poses.csv"),
                  meta={"source": SOURCE, "radius_mm": RADIUS_MM,
                        "axial": segment.AXIAL_DEFAULT}) as rec:
    src2 = sources.open_source(SOURCE)
    i = 0
    while True:
        item = src2.read()
        if item is None or (MAX_FRAMES and i >= MAX_FRAMES):
            break
        t_cap, frame = item
        pose = est.update(frame, t=t_cap)
        state = filt.update(pose, t=t_cap)
        rec.write(pose, t_capture=t_cap, frame_index=i,
                  velocity=None if state is None else state[1])
        if pose is None:
            lost += 1
        else:
            t_ms.append(pose.t_total_ms)
            rows.append((t_cap, *pose.xyz_mm, pose.theta_deg, pose.phi_deg,
                         pose.psi_deg, pose.ambiguity_margin_deg, pose.fit_rms_px))
        i += 1

print(f"{i} frames, {len(rows)} detected, {lost} lost ({100 * lost / max(i, 1):.1f}%)")
print(f"compute {np.median(t_ms):.2f} ms median, {np.percentile(t_ms, 95):.2f} ms p95"
      f"  -> {1e3 / np.median(t_ms):.0f} Hz capable")

## 4. The six channels

Position in three axes, then tilt / azimuth / in-plane angle. `psi` is the image
ellipse's major-axis angle — geometrically tied to `phi`, carried separately as a
consistency check. **It is not spin**: the robot turns at 310–350 Hz against a
camera an order of magnitude slower, so blade phase aliases beyond rescue and roll
is not recoverable at all.

In [ ]:
a = np.array(rows, dtype=float)
t = a[:, 0] - a[0, 0]
names = ["x (mm)", "y (mm)", "z (mm)", "tilt θ (deg)", "azimuth φ (deg)", "ψ (deg)"]

fig, axes = plt.subplots(3, 2, figsize=(11, 7), sharex=True)
for k, (ax, nm) in enumerate(zip(axes.T.ravel(), names)):
    ax.plot(t, a[:, 1 + k], lw=0.9)
    ax.set_ylabel(nm); ax.grid(alpha=0.3)
for ax in axes[-1]:
    ax.set_xlabel("time (s)")
plt.tight_layout(); plt.show()

## 5. Which frames should you not believe?

Two per-frame numbers say so without any ground truth:

- **`fit_rms_px`** — how far the silhouette departs from *any* ellipse, i.e. how
  much rod/magnet contamination is in the outline.
- **`ambiguity_margin_deg`** — how far apart the two candidate poses were. Small
  means the branch choice barely mattered; large means a wrong pick is a large
  error, and prior-free that pick is close to a coin toss.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
ax[0].hist(t_ms, bins=40, color="0.4"); ax[0].set_xlabel("compute (ms)")
ax[0].axvline(1e3 / 240, color="crimson", label="240 fps budget"); ax[0].legend()
ax[1].hist(a[:, 8], bins=40, color="0.4"); ax[1].set_xlabel("fit rms (px)")
ax[2].hist(a[:, 7], bins=40, color="0.4"); ax[2].set_xlabel("ambiguity margin (deg)")
for x in ax: x.grid(alpha=0.3)
plt.tight_layout(); plt.show()

sus = np.argsort(a[:, 8])[-3:][::-1]
print("worst fit_rms frames (indices into the detected set):", sus)

## 6. What the residual *cannot* be

`bounds.py` gives the Cramér–Rao floor for this geometry. Comparing a measured
residual against it is what separates "the solver is sloppy" from "the information
is not in the image" — and here it is decisively the latter: the measured boundary
scatter is ~23× the photon bound, so better sensors, longer exposures and finer
sub-pixel interpolation are all inert (lecture notes §13.8).

In [ ]:
z_med = float(np.median(a[:, 3]))
b = bounds.budget(z_med, RADIUS_MM, K, tilt_deg=float(np.median(a[:, 4])))
print(f"at the observed median range {z_med:.0f} mm:")
print(f"  rim spans          {2 * b['semi_major_px']:.0f} px")
print(f"  photon edge floor  {b['photon_sigma_px']:.4f} px")
print(f"  pixel quantisation {b['quantisation_sigma_px']:.4f} px")
print(f"  depth penalty      {b['depth_lateral_ratio']:.1f}x lateral")